In [1]:
"""
NOW WE HAVE THIS Qij*, Ai generated and we are sending this to the KB

WE GET BACK: documents d1, ... d5

WE NEED TO CHECK: IS THE RIGHT ANSWER A IN d1, .. d5

For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.
"""

"\nNOW WE HAVE THIS Qij*, Ai generated and we are sending this to the KB\n\nWE GET BACK: documents d1, ... d5\n\nWE NEED TO CHECK: IS THE RIGHT ANSWER A IN d1, .. d5\n\nFor search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.\n"

In [2]:
# For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.

# read data: 

import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth

[{'question': 'Is it still possible to join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Can I start the course after it has already begun?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course now, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': "What do I need to do to qualify for the certificate if I'm joining late?",
  'document': '74eb249bbf'},
 {'question': 'Are project submissions only accepted during a certain period for the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I registered for the LLM Zoomcamp — when should I get a confirmation email?',
  'document': '977bf7786c'},
 {'question': 'Do I actually need to wait for approval before starting the course and submitting homework?',
  'document': '977bf7786c'},
 {'question': 'Is there some kind of accepted list for the LLM Zoomcamp registration?',
  'document': '977bf7786c'},
 {'question': "Why do we have to register for the course if it isn't 

In [3]:
# Now we need to define the search engine

# using minsearch

# ingest has this build index that is used for text search e.g. like index.searh(...)

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
# we want to make our framework modular 
# we shouldn't really care which search function is behind , as long as this funtion returns some results
# we call it deliberately text_search and later want to call it vector_search

def text_search(query):
    # the question boost is 3.0 if the user question matches exactly the faq then its more important
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [5]:
q = ground_truth[0]
q

{'question': 'Is it still possible to join the course if I found it late?',
 'document': '74eb249bbf'}

In [6]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [7]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
9f689c185f == 74eb249bbf: False


In [8]:
# this gives the relevance matrix = table:

# this matrix can help us to calculate how good some things work

In [9]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [10]:
# we need to do this for all the documents so we ut this into function

def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [11]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Is it still possible to join the course if I found it late?


[1, 0, 0, 0, 0]

In [16]:
q = ground_truth[12]
print(q["question"])
compute_relevance_text(q)


Can students get access to the Zoom for office hours, or is it only for the course staff?


[1, 0, 0, 0, 0]

In [17]:
# lets assume there will also be queries where we can't actually retrieve 
# [0, 0, 0, 0, 0]



In [18]:
# now this is function to do it for all questions 

from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [19]:
# for first 15 
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [20]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [21]:
# make it generic so we can evaluate vector search, hybrid search, or another retrieval method
# but the relevance logic stays the same 

def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance



In [22]:
# for all questions in the ground truth data 

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [23]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [ ]:
# Evaluation metrics computation 

# SEARCH EVALUATION METRICS 


In [ ]:
# Hit rate: 

# for every row that has a one we count 1 

# in our example we have 15 examples and our hit rate is 14 

# maybe our examples are not realistic, a problem with sythetic data set
# they doesn't represent how people query 

# in our case we continue, with had hitrate 

14/15 

# 93 percent 

0.9333333333333333

In [25]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt


cnt / len(example)
# 0.933

0.9333333333333333